# EnerQit: a reproducible inverse-engineering study

Synthetic evidence only. Install the package in this kernel first (`pip install -e ".[examples]"` from the repository root). This notebook compares a physical classical estimator and discrete QUBO solvers; it does not claim a quantum advantage.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from enerqit.datasets import coupled_inventory, fixed_array
from enerqit.inference import fit_orientation
from enerqit.pvsystem import fixed_ac
from enerqit.solvers import exact, group_exact, simulated_annealing, simulated_bifurcation

## 1. Recover known synthetic geometry
Weather, location, capacities and loss-model assumptions are required. AC alone is not enough.

In [ ]:
frame, truth = fixed_array(seed=42)
fit = fit_orientation(frame, 35, -5)
print(fit.to_json())

In [ ]:
prediction = fixed_ac(frame, 35, -5, **fit.parameters)
fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(frame.index[:96], frame.power_ac.iloc[:96] / 1000, ".", label="Synthetic")
ax.plot(frame.index[:96], prediction[:96] / 1000, label="Fit")
ax.set_ylabel("AC power (kW)")
ax.legend()
fig.autofmt_xdate()
plt.show()

## 2. Compare solvers on coupled configuration hypotheses
This separate controlled surrogate links choices with a known inventory. Classical feasible enumeration exploits the native structure. Always inspect feasibility and physical residual, not only QUBO energy.

In [ ]:
problem, planted, templates, observations = coupled_inventory(seed=42)
for solver in (exact, group_exact, simulated_annealing, simulated_bifurcation):
    result = solver(problem)
    print(
        result.method,
        "energy",
        result.energy,
        "feasible",
        result.feasible,
        "seconds",
        result.elapsed_s,
        "signal RMSE",
        np.sqrt(np.mean((templates @ result.x - observations) ** 2)),
    )

## 3. Migration to quantum hardware
The same binary model can be exported. The built-in QAOA helper is classical statevector simulation, limited to small cases. Real hardware experiments need separate cost, embedding/transpilation and noise analysis.

In [ ]:
from enerqit.solvers.quantum import ising_payload

payload = ising_payload(problem)
print(payload.keys())
# With the quantum extra installed:
# from enerqit.solvers.quantum import to_dimod, to_qiskit
# bqm = to_dimod(problem)
# operator = to_qiskit(problem)

## What the result establishes
The synthetic recovery is an implementation check. It does not establish field accuracy, OEM certification, global identifiability, or solver advantage. See the seven gallery examples for tracker settings, IV/MPPT reconstruction, inverter curves and versioned daily evidence.